In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, confusion_matrix, classification_report
import pickle
from tqdm.auto import tqdm
import pandas as pd
import seaborn as sns
import datetime
import sys

# 导入您的自定义数据加载器
# 假设该文件名是brain_voxel_dataloader.py
from brain_voxel_dataloader import BrainVoxelDataLoader

# 定义用于检查数据是否标准化的函数
def check_standardization(data, threshold=0.1, sample_size=1000):
    """
    检查数据是否已标准化
    
    参数:
        data: numpy数组或PyTorch张量，形状为 [n_samples, n_features]
        threshold: 均值和标准差允许的偏差阈值
        sample_size: 检查的样本数量，如果数据很大，我们只检查一部分
    
    返回:
        bool: 数据是否已标准化
    """
    # 如果是PyTorch张量，转换为numpy数组
    if isinstance(data, torch.Tensor):
        data = data.cpu().numpy()
    
    # 如果数据很大，随机抽样
    if len(data) > sample_size:
        indices = np.random.choice(len(data), sample_size, replace=False)
        data = data[indices]
    
    # 计算每个特征的均值和标准差
    means = np.mean(data, axis=0)
    stds = np.std(data, axis=0)
    
    # 检查均值是否接近0，标准差是否接近1
    mean_close_to_zero = np.all(np.abs(means) < threshold)
    std_close_to_one = np.all(np.abs(stds - 1.0) < threshold)
    
    return mean_close_to_zero and std_close_to_one

# 定义模型类
class DenseModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=4096, num_classes=102, dropout_rate=0.5):
        super(DenseModel, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.layer2 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.layer3 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.layer4 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.output_layer = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return self.output_layer(x)

# 训练模型函数
def train_model(train_loader, val_loader, input_dim=341, num_classes=102, device='cuda', 
                hidden_dim=4096, dropout_rate=0.5, learning_rate=0.00001, weight_decay=0.00001, 
                no_epochs=25, checkpoint_dir=None):
    # 创建模型
    model = DenseModel(
        input_dim=input_dim, 
        hidden_dim=hidden_dim, 
        num_classes=num_classes, 
        dropout_rate=dropout_rate
    ).to(device)
    
    # 定义损失函数和优化器
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        model.parameters(), 
        lr=learning_rate, 
        weight_decay=weight_decay
    )
    
    # 初始化训练历史
    history = {
        'train_loss': [], 'train_acc': [], 'train_f1': [],
        'val_loss': [], 'val_acc': [], 'val_f1': []
    }
    
    # 训练模型
    for epoch in range(no_epochs):
        # 训练阶段
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        all_train_preds = []
        all_train_targets = []
        
        # 进度条
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{no_epochs} [Train]')
        
        for batch in pbar:
            # 处理数据中包含病人ID的情况
            if len(batch) == 3:  # (features, labels, patient_ids)
                inputs, labels, _ = batch
            else:  # (features, labels)
                inputs, labels = batch
            
            # 检查输入是否已经是PyTorch张量
            if isinstance(inputs, torch.Tensor):
                inputs = inputs.to(dtype=torch.float32, device=device)
            else:
                inputs = torch.tensor(inputs, dtype=torch.float32, device=device)
            
            # 处理one-hot标签
            if labels.ndim > 1 and labels.shape[1] > 1:  # one-hot格式
                if isinstance(labels, torch.Tensor):
                    labels_tensor = labels
                else:
                    labels_tensor = torch.tensor(labels)
                target_indices = torch.argmax(labels_tensor, dim=1).to(device)
            else:  # 索引格式
                if isinstance(labels, torch.Tensor):
                    target_indices = labels.to(dtype=torch.long, device=device)
                else:
                    target_indices = torch.tensor(labels, dtype=torch.long, device=device)
            
            # 前向传播
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, target_indices)
            
            # 反向传播和优化
            loss.backward()
            optimizer.step()
            
            # 统计
            train_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            train_total += target_indices.size(0)
            train_correct += (predicted == target_indices).sum().item()
            
            # 收集预测和目标用于计算F1
            all_train_preds.extend(predicted.cpu().numpy())
            all_train_targets.extend(target_indices.cpu().numpy())
            
            # 更新进度条
            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{train_correct/train_total:.4f}'
            })
        
        train_loss = train_loss / train_total
        train_acc = train_correct / train_total
        train_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
        
        # 验证阶段
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        all_val_preds = []
        all_val_targets = []
        
        with torch.no_grad():
            # 进度条
            pbar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{no_epochs} [Val]')
            
            for batch in pbar:
                # 处理数据中包含病人ID的情况
                if len(batch) == 3:  # (features, labels, patient_ids)
                    inputs, labels, _ = batch
                else:  # (features, labels)
                    inputs, labels = batch
                
                # 检查输入是否已经是PyTorch张量
                if isinstance(inputs, torch.Tensor):
                    inputs = inputs.to(dtype=torch.float32, device=device)
                else:
                    inputs = torch.tensor(inputs, dtype=torch.float32, device=device)
                
                # 处理one-hot标签
                if labels.ndim > 1 and labels.shape[1] > 1:  # one-hot格式
                    if isinstance(labels, torch.Tensor):
                        labels_tensor = labels
                    else:
                        labels_tensor = torch.tensor(labels)
                    target_indices = torch.argmax(labels_tensor, dim=1).to(device)
                else:  # 索引格式
                    if isinstance(labels, torch.Tensor):
                        target_indices = labels.to(dtype=torch.long, device=device)
                    else:
                        target_indices = torch.tensor(labels, dtype=torch.long, device=device)
                
                # 前向传播
                outputs = model(inputs)
                loss = criterion(outputs, target_indices)
                
                # 统计
                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs.data, 1)
                val_total += target_indices.size(0)
                val_correct += (predicted == target_indices).sum().item()
                
                # 收集预测和目标用于计算F1
                all_val_preds.extend(predicted.cpu().numpy())
                all_val_targets.extend(target_indices.cpu().numpy())
                
                # 更新进度条
                pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'acc': f'{val_correct/val_total:.4f}'
                })
        
        val_loss = val_loss / val_total
        val_acc = val_correct / val_total
        val_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
        
        # 记录历史
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['train_f1'].append(train_f1)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        
        # 打印当前epoch的结果
        print(f"Epoch {epoch+1}/{no_epochs}")
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Train F1: {train_f1:.4f}")
        print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")
        
        # 保存检查点
        if checkpoint_dir:
            checkpoint_path = os.path.join(checkpoint_dir, f'checkpoint_epoch_{epoch+1}.pth')
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': train_loss,
                'val_loss': val_loss,
                'train_acc': train_acc,
                'val_acc': val_acc,
                'train_f1': train_f1,
                'val_f1': val_f1,
                'history': history
            }, checkpoint_path)
            print(f"检查点已保存到: {checkpoint_path}")
    
    return model, history

# 评估模型函数
def evaluate_model(model, test_loader, device='cuda'):
    model.eval()
    test_loss = 0.0
    test_correct = 0
    test_total = 0
    all_preds = []
    all_targets = []
    
    criterion = nn.CrossEntropyLoss()
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc='Testing'):
            # 处理数据中包含病人ID的情况
            if len(batch) == 3:  # (features, labels, patient_ids)
                inputs, labels, _ = batch
            else:  # (features, labels)
                inputs, labels = batch
            
            # 检查输入是否已经是PyTorch张量
            if isinstance(inputs, torch.Tensor):
                inputs = inputs.to(dtype=torch.float32, device=device)
            else:
                inputs = torch.tensor(inputs, dtype=torch.float32, device=device)
            
            # 处理one-hot标签
            if labels.ndim > 1 and labels.shape[1] > 1:  # one-hot格式
                if isinstance(labels, torch.Tensor):
                    labels_tensor = labels
                else:
                    labels_tensor = torch.tensor(labels)
                target_indices = torch.argmax(labels_tensor, dim=1).to(device)
            else:  # 索引格式
                if isinstance(labels, torch.Tensor):
                    target_indices = labels.to(dtype=torch.long, device=device)
                else:
                    target_indices = torch.tensor(labels, dtype=torch.long, device=device)
            
            # 前向传播
            outputs = model(inputs)
            loss = criterion(outputs, target_indices)
            
            # 统计
            test_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            test_total += target_indices.size(0)
            test_correct += (predicted == target_indices).sum().item()
            
            # 收集预测和目标
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(target_indices.cpu().numpy())
    
    test_loss = test_loss / test_total
    test_acc = test_correct / test_total
    test_f1 = f1_score(all_targets, all_preds, average='macro')
    
    print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f} | Test F1: {test_f1:.4f}")
    
    # 计算分类报告
    report = classification_report(all_targets, all_preds)
    print("Classification Report:")
    print(report)
    
    return test_loss, test_acc, test_f1, all_preds, all_targets

    

In [ ]:

# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

# 设置超参数
batch_size = 128
input_dim = 341
num_classes = 102

# 生成时间戳，用于保存文件
timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_dir = f'./output/{timestamp}'
os.makedirs(output_dir, exist_ok=True)

# 初始化数据加载器 - 启用交叉验证
data_loader = BrainVoxelDataLoader(
    base_dir='/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/reorganized_fold_data',  # 修改为实际路径
    cross_validation=True,  # 启用交叉验证
    cv_fold=30,  # 30折交叉验证
    standardize=True,  # 启用标准化
    output_dir=output_dir,  # 使用带时间戳的输出目录
    shuffle=True,  # 打乱训练集
    random_seed=42,  # 随机种子
    save_scaler=True  # 保存标准化器
)

# 获取测试集
test_dataset = data_loader.get_test_dataset()

# 定义StandardizedDataset类用于手动标准化
class StandardizedDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, scaler, already_standardized=False):
        self.dataset = dataset
        self.scaler = scaler
        self.already_standardized = already_standardized
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        data = self.dataset[idx]
        
        if len(data) == 3:  # features, labels, patient_ids
            features, labels, patient_ids = data
            
            # 确保features是numpy数组
            if isinstance(features, torch.Tensor):
                features = features.numpy()
            
            if not self.already_standardized:
                # 如果是1D数组，先reshape为2D
                if features.ndim == 1:
                    features = self.scaler.transform(features.reshape(1, -1)).flatten()
                else:
                    features = self.scaler.transform(features)
            
            return features, labels, patient_ids
        else:  # features, labels
            features, labels = data
            
            # 确保features是numpy数组
            if isinstance(features, torch.Tensor):
                features = features.numpy()
            
            if not self.already_standardized:
                # 如果是1D数组，先reshape为2D
                if features.ndim == 1:
                    features = self.scaler.transform(features.reshape(1, -1)).flatten()
                else:
                    features = self.scaler.transform(features)
            
            return features, labels

# 执行30折交叉验证
cv_results = []
for fold in range(30):
    print(f"\n开始交叉验证第 {fold+1}/30 折")
    
    # 获取当前折的训练集和验证集
    train_dataset, val_dataset = data_loader.get_cv_fold(fold)
    
    print(f"训练集大小: {len(train_dataset)}")
    print(f"验证集大小: {len(val_dataset)}")
    print(f"测试集大小: {len(test_dataset)}")
    
    # 检查数据集是否已标准化
    print("检查数据集是否已标准化...")
    
    # 随机抽样检查标准化
    sample_size = min(1000, len(train_dataset))
    
    # 获取数据样本进行检查
    train_features = np.array([train_dataset[i][0] for i in range(sample_size)])
    val_features = np.array([val_dataset[i][0] for i in range(min(sample_size, len(val_dataset)))])
    test_features = np.array([test_dataset[i][0] for i in range(min(sample_size, len(test_dataset)))])
    
    train_standardized = check_standardization(train_features)
    val_standardized = check_standardization(val_features)
    test_standardized = check_standardization(test_features)
    
    print(f"训练集是否已标准化: {train_standardized}")
    print(f"验证集是否已标准化: {val_standardized}")
    print(f"测试集是否已标准化: {test_standardized}")
    
    # 如果数据没有标准化，手动标准化
    if not (train_standardized and val_standardized and test_standardized):
        print("检测到数据未标准化，正在进行手动标准化...")
        
        # 确保我们有标准化器
        if hasattr(data_loader, 'scaler') and data_loader.scaler is not None:
            scaler = data_loader.scaler
            
            # 创建标准化的数据集
            if not train_standardized:
                train_dataset = StandardizedDataset(train_dataset, scaler, False)
                print("已手动标准化训练集")
            
            if not val_standardized:
                val_dataset = StandardizedDataset(val_dataset, scaler, False)
                print("已手动标准化验证集")
            
            if not test_standardized:
                test_dataset = StandardizedDataset(test_dataset, scaler, False)
                print("已手动标准化测试集")
            
            # 再次检查标准化
            train_features = np.array([train_dataset[i][0] for i in range(sample_size)])
            val_features = np.array([val_dataset[i][0] for i in range(min(sample_size, len(val_dataset)))])
            test_features = np.array([test_dataset[i][0] for i in range(min(sample_size, len(test_dataset)))])
            
            train_standardized = check_standardization(train_features)
            val_standardized = check_standardization(val_features)
            test_standardized = check_standardization(test_features)
            
            print(f"手动标准化后，训练集是否已标准化: {train_standardized}")
            print(f"手动标准化后，验证集是否已标准化: {val_standardized}")
            print(f"手动标准化后，测试集是否已标准化: {test_standardized}")
            
            # 如果仍然有数据集未标准化，保存样本并终止程序
            if not (train_standardized and val_standardized and test_standardized):
                print("错误: 手动标准化后仍有数据集未标准化!")
                
                # 保存未标准化的数据样本用于调试
                debug_dir = os.path.join(output_dir, f'debug_fold_{fold+1}')
                os.makedirs(debug_dir, exist_ok=True)
                
                np.save(os.path.join(debug_dir, 'train_features_sample.npy'), train_features)
                np.save(os.path.join(debug_dir, 'val_features_sample.npy'), val_features)
                np.save(os.path.join(debug_dir, 'test_features_sample.npy'), test_features)
                
                # 保存一些统计信息
                stats = {
                    'train_mean': np.mean(train_features, axis=0),
                    'train_std': np.std(train_features, axis=0),
                    'val_mean': np.mean(val_features, axis=0),
                    'val_std': np.std(val_features, axis=0),
                    'test_mean': np.mean(test_features, axis=0),
                    'test_std': np.std(test_features, axis=0),
                    'scaler_mean': scaler.mean_,
                    'scaler_scale': scaler.scale_
                }
                
                with open(os.path.join(debug_dir, 'stats.pkl'), 'wb') as f:
                    pickle.dump(stats, f)
                
                print(f"未标准化的数据样本已保存到: {debug_dir}")
                print("程序终止。")
                sys.exit(1)
        else:
            print("错误：无法获取标准化器！程序终止。")
            sys.exit(1)
    
    # 创建数据加载器
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    
    # 为当前折创建目录
    fold_dir = os.path.join(output_dir, f'fold_{fold+1}')
    os.makedirs(fold_dir, exist_ok=True)
    
    # 训练模型，使用fold_dir作为检查点目录
    model, history = train_model(
        train_loader, 
        val_loader, 
        input_dim=input_dim, 
        num_classes=num_classes, 
        device=device,
        checkpoint_dir=fold_dir
    )
    
    # 评估模型
    test_loss, test_acc, test_f1, y_pred, y_true = evaluate_model(model, test_loader, device=device)
    
    # 保存最终模型
    torch.save(model.state_dict(), os.path.join(fold_dir, 'final_model.pth'))
    
    # 将当前折的结果添加到交叉验证结果中
    fold_result = {
        'fold': fold + 1,
        'test_loss': test_loss,
        'test_acc': test_acc,
        'test_f1': test_f1,
        'history': history
    }
    cv_results.append(fold_result)
    
    # 保存当前折的结果
    with open(os.path.join(fold_dir, f'fold_{fold+1}_results.pkl'), 'wb') as f:
        pickle.dump(fold_result, f)
    
    # 保存当前折的标准化器（如果可用）
    if hasattr(data_loader, 'scaler') and data_loader.scaler is not None:
        scaler_path = os.path.join(fold_dir, f'scaler_fold_{fold+1}.pkl')
        with open(scaler_path, 'wb') as f:
            pickle.dump(data_loader.scaler, f)
        print(f"标准化器已保存到: {scaler_path}")

# 保存所有交叉验证结果
cv_results_path = os.path.join(output_dir, f'cv_results_{timestamp}.pkl')
with open(cv_results_path, 'wb') as f:
    pickle.dump(cv_results, f)
print(f"交叉验证结果已保存到: {cv_results_path}")

# 计算并显示交叉验证平均结果
mean_test_loss = np.mean([result['test_loss'] for result in cv_results])
mean_test_acc = np.mean([result['test_acc'] for result in cv_results]) * 100  # 转换为百分比
mean_test_f1 = np.mean([result['test_f1'] for result in cv_results])
std_test_loss = np.std([result['test_loss'] for result in cv_results])
std_test_acc = np.std([result['test_acc'] for result in cv_results]) * 100  # 转换为百分比
std_test_f1 = np.std([result['test_f1'] for result in cv_results])

print(f"\n交叉验证平均测试损失: {mean_test_loss:.4f} ± {std_test_loss:.4f}")
print(f"交叉验证平均测试准确率: {mean_test_acc:.2f}% ± {std_test_acc:.2f}%")
print(f"交叉验证平均测试F1分数: {mean_test_f1:.4f} ± {std_test_f1:.4f}")

# 保存平均结果
summary = {
    'mean_test_loss': mean_test_loss,
    'mean_test_acc': mean_test_acc,
    'mean_test_f1': mean_test_f1,
    'std_test_loss': std_test_loss,
    'std_test_acc': std_test_acc,
    'std_test_f1': std_test_f1
}

with open(os.path.join(output_dir, 'summary.pkl'), 'wb') as f:
    pickle.dump(summary, f)

# 绘制交叉验证结果
plt.figure(figsize=(15, 10))

# 准确率
plt.subplot(2, 2, 1)
plt.errorbar(
    range(1, len(cv_results) + 1), 
    [result['test_acc'] * 100 for result in cv_results],  # 转换为百分比
    yerr=std_test_acc, 
    fmt='o-', 
    capsize=5
)
plt.axhline(y=mean_test_acc, color='r', linestyle='--', label=f'Mean: {mean_test_acc:.2f}%')
plt.title('Cross-Validation Results: Test Accuracy')
plt.xlabel('Fold')
plt.ylabel('Accuracy (%)')
plt.grid(True)
plt.legend()

# F1分数
plt.subplot(2, 2, 2)
plt.errorbar(
    range(1, len(cv_results) + 1), 
    [result['test_f1'] for result in cv_results],
    yerr=std_test_f1, 
    fmt='o-', 
    capsize=5,
    color='g'
)
plt.axhline(y=mean_test_f1, color='r', linestyle='--', label=f'Mean: {mean_test_f1:.4f}')
plt.title('Cross-Validation Results: Test F1 Score')
plt.xlabel('Fold')
plt.ylabel('F1 Score')
plt.grid(True)
plt.legend()

# 损失
plt.subplot(2, 2, 3)
plt.errorbar(
    range(1, len(cv_results) + 1), 
    [result['test_loss'] for result in cv_results],
    yerr=std_test_loss, 
    fmt='o-', 
    capsize=5,
    color='b'
)
plt.axhline(y=mean_test_loss, color='r', linestyle='--', label=f'Mean: {mean_test_loss:.4f}')
plt.title('Cross-Validation Results: Test Loss')
plt.xlabel('Fold')
plt.ylabel('Loss')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(output_dir, f'cv_results_{timestamp}.png'))
print(f"交叉验证结果图已保存到: {os.path.join(output_dir, f'cv_results_{timestamp}.png')}")